In [ ]:
import astropy.units as u
from astropy.io import fits
from astropy.table import QTable
from astropy.time import Time
from astropy.utils.data import download_file
from astropy.coordinates import (SkyCoord, Distance, Galactic, 
                                 EarthLocation, AltAz)
import astropy.coordinates as coord
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib as mpl
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
%matplotlib inline 

In [ ]:
MS_WDs = pd.read_csv('/Volumes/T7/research/2024_2025/Code/mass_calcs/MS-WD/MSWD_sample.xls') #read in your csv which contains either Gaia IDs to find orbital solutions from NSS or your own orbital solutions
MS_WDs

In [ ]:
source_id = MS_WDs['gaia_source_id'].to_numpy(dtype='int') # need Gaia IDs to search NSS, make an array of them

In [ ]:
gaiadr3_table = Gaia.load_table('gaiadr3.nss_two_body_orbit') #this just tells you what all the columns in Gaia NSS are. you can also search up the Gaia AIP
print(gaiadr3_table)                                          #for NSS https://gaia.aip.de/metadata/gaiadr3/nss_two_body_orbit/ to get a better understanding

for column in gaiadr3_table.columns:
  print(column.name)

In [ ]:
MS_WDs_query = "SELECT * \
FROM gaiadr3.gaia_source AS dr3 \
JOIN gaiadr3.nss_two_body_orbit USING (source_id) \
WHERE (dr3.source_id in" + str(tuple(source_id))+")"

#here you're searching the Gaia NSS database, checking if your sources have been identified as doubles by the NSS pipeline. I'll go line by line

#so you're using Gaia ADQL (a Query Language) to SELECT all the stars that fit the criteria you are about to impose
#FROM gaiadr3.gaia_source AS dr3: we are using the source_id from the gaia_source database to JOIN the information form Gaia NSS by the Gaia source ID
#WHERE (dr3.source_id in" + str(tuple(source_id))+")" : that's our array we set

MS_WDs_job = Gaia.launch_job(MS_WDs_query) #this actually sends the message and criteria we want to the Gaia servers
MS_WDs_data = MS_WDs_job.get_results() #this gives us our results

In [ ]:
MS_WDs_data_csv = MS_WDs_data.to_pandas()
MS_WDs_data_csv

In [ ]:
MS_WDs_data_csv['nss_solution_type'] #for the MS-WD sample, you'll see they're all astrometric solutions (orbital and astrospectroSB1)
                                     #basically, Gaia instruments are sensitive enough to resolve the photometric wobble of these systems and "see"
                                     #the wobble, providing an astrometric solution. astrospectroSB1 is a combination of astrometric and a spectroscopic
                                     #solution

In [ ]:
Gaia_SB1 = MS_WDs_data_csv[MS_WDs_data_csv['nss_solution_type']=='SB1']

Gaia_SB2 = MS_WDs_data_csv[MS_WDs_data_csv['nss_solution_type']=='SB2']

astrometric_mask = (MS_WDs_data_csv['nss_solution_type']=='Orbital') | (MS_WDs_data_csv['nss_solution_type']=='AstroSpectroSB1')
Gaia_astrometric = MS_WDs_data_csv[astrometric_mask]

In [ ]:
Gaia_astrometric

In [ ]:
#the majority of these functions come from Halbwachs 2023: https://ui.adsabs.harvard.edu/abs/2023A%26A...674A...9H/abstract
#Context: Thiele Innes elements are basically parameters that linearize the equations of motion of a companion star relative to its primary in the sky

def u(A,B,F,G):
    u = (A**2+B**2+F**2+G**2)/2
    return u #this is from equation A.2 in Halbwachs 2023. it is one part of the semi-major axis calculations using the Thiele Innes elements

def v(A,B,F,G):
    v = A*G - B*F
    return v #this is from equation A.2 in Halbwachs 2023. it the other part of the semi-major axis calculations using the Thiele Innes elements

def semi_major(u, v):
    a = np.sqrt(u+np.sqrt((u+v)*(u-v)))
    return a #this calculates the semi-major axis of the orbit

def halb_mass_func(a, P, w):
    f = ((a**3)*(365.25**2))/((P**2)*(w**3))
    return f #this is equation 13 in Halbwachs 2023, calcualting the mass function of a system based on astrometry. it is superior to the binary mass function as it does not have a sin(i) dependence

def bin_mass_func(P, K, e):
    f = (4.34389*(10**(-17)))*((P*K**3)/(2*np.pi*6.6743*10**-11))*((1-e**2)**(3/2)) #86400 seconds in a day, m^3 to km^3 is 1E-9, mass of sun is 1.989E30 kg. mass func in units of solar masses
    return f #this is the binary mass function, described by Kepler's 3rd law, used for spectroscopic solutions. it's not as as awesome as the Halbwachs one because it has a sin(i) dependence. based on spectroscopy

def comp_mass(M1, fm): #defining a function to solve the cubic equation for
    coeff = [1, -fm, ((-fm)*2*(M1)), ((-fm)*(M1**2))] #the secondary mass in an SB1
    M2 = np.roots(coeff) #it takes the primary mass and mass function and 
    return M2 #calculates the secondary mas

def sb2_mass(K1,K2,M1):
    q = K1/K2 #in SB2 systems, the mass ratio is inversely proportional to the semi-amplitude ratio!
    M2 = M1*q
    return M2

In [ ]:
a = semi_major(u(Gaia_astrometric['a_thiele_innes'], Gaia_astrometric['b_thiele_innes'], Gaia_astrometric['f_thiele_innes'], Gaia_astrometric['g_thiele_innes']),v(Gaia_astrometric['a_thiele_innes'], Gaia_astrometric['b_thiele_innes'], Gaia_astrometric['f_thiele_innes'], Gaia_astrometric['g_thiele_innes']))
print(a, 'in milli-arcseconds')

In [ ]:
mass_func_orbital = halb_mass_func(a, Gaia_astrometric['period'], Gaia_astrometric['parallax'])
mass_func_orbital = np.array(mass_func_orbital, dtype=float)
print(mass_func_orbital, 'in solar masses')

In [ ]:
Gaia_astrometric['fm'] = mass_func_orbital #adding the actual mass function to the larger table

In [ ]:
masses = np.arange(0.5,1.05, 0.05) #arbitrary mass range for the primary mass assumptions in order to calculate a secondary
masses

In [ ]:
for j in range(len(masses)):
    cubic_soln = [] #creates and/or resets an array that stores the real solution to the cubic function
    for i in range(len(mass_func_orbital)):
        M2_astro=comp_mass(masses[j],mass_func_orbital[i])[0] #calculates the mass of the companion, iterating through assumed primary masses, and picks only the REAL solution in the cubic
        cubic_soln.append(M2_astro) #append the solution here
    cubic_soln = np.array(cubic_soln, dtype=str) #change it to a string so we can get rid of the complex part that comes with the solution
    M2_soln_astro = [x[1:16] for x in cubic_soln] #remove the complex part
    M2_soln_astro = np.array(M2_soln_astro, dtype=np.float32) #convert it to a float
    M2_soln_astro
    Gaia_astrometric['M2 (M1=' + str(masses[j]) + 'Msun)'] = M2_soln_astro

In [ ]:
Gaia_astrometric

In [ ]:
cm = mpl.colormaps['viridis'] 

In [ ]:
plt.scatter(Gaia_astrometric['period'][(Gaia_astrometric['nss_solution_type']=='Orbital') | (Gaia_astrometric['nss_solution_type']=='AstroSpectroSB1')], Gaia_astrometric['eccentricity'][(Gaia_astrometric['nss_solution_type']=='Orbital') | (Gaia_astrometric['nss_solution_type']=='AstroSpectroSB1')], c = Gaia_astrometric['M2 (M1=1.0000000000000004Msun)'][(Gaia_astrometric['nss_solution_type']=='Orbital') | (Gaia_astrometric['nss_solution_type']=='AstroSpectroSB1')], cmap=cm, vmin = 0.35, vmax=1.0, marker='s', s=100, linewidth=1, edgecolors='black', label='orbital')
plt.xlabel(r'$log(P_{orb})$ (days)')
plt.ylabel('eccentricity')
plt.title(r'Minimum Gaia NSS secondary mass solns ($M_1 \approx 1.0M_\odot$)')
cbar = plt.colorbar()
cbar.solids.set_edgecolor("face")
cbar.ax.set_ylabel(r'$M_2$ ($M_\odot$)')
cbar.minorticks_on()
plt.xscale('log')
plt.legend()
plt.draw()

In [ ]:
MS_WDs.rename(columns={'gaia_source_id':'source_id'}, inplace = True)

In [ ]:
MS_fits_table = pd.merge(MS_WDs, Gaia_astrometric[['source_id', 'fm', 'period', 'eccentricity']], on='source_id')
MS_fits_table

In [ ]:
cubic_soln = []

In [ ]:
for i in range(len(mass_func_orbital)):
    M2_fit_astro=comp_mass(MS_fits_table['fit_MS_mass'][i],MS_fits_table['fm'][i])[0] #SAME THING but now we use the best fit or known masses of the primary for the secondary calculation
    cubic_soln.append(M2_fit_astro)
cubic_soln = np.array(cubic_soln, dtype=str)
M2_fit_soln_astro = [x[1:16] for x in cubic_soln]
M2_fit_soln_astro = np.array(M2_fit_soln_astro, dtype=np.float32)
M2_fit_soln_astro
MS_fits_table['M2 (Msun)'] = M2_fit_soln_astro

In [ ]:
MS_fits_table